# Height-map target extraction exploration

Exploratory, non-production method-development notebook for NSF Future Manufacturing Data Challenge Person B work.

Scope: Tracks 8, 10, and 14 only. Track 21 is intentionally sealed and is not loaded or analyzed.

This notebook compares detrending, adaptive height candidate segmentation, continuity-aware component selection, and gradient boundary refinement before freezing a production target extractor.

In [ ]:
from pathlib import Path
from datetime import datetime
import sys, json, math, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def find_project_dir(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    here = Path(__file__).resolve() if '__file__' in globals() else None
    if here is not None:
        candidates += [here.parent, *here.parents]
    for p in candidates:
        if (p / 'src' / 'nsf_fmrg_data.py').exists() and (p / 'data' / 'raw' / 'height_maps').exists():
            return p
    raise RuntimeError('Could not resolve project root containing src/nsf_fmrg_data.py and data/raw/height_maps')

PROJECT_DIR = find_project_dir()
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nsf_fmrg_data import load_wyko_asc, robust_plane_detrend

HEIGHT_DIR = PROJECT_DIR / 'data' / 'raw' / 'height_maps'
TRACK_IDS = [8, 10, 14]  # Track 21 is sealed for this experiment.
REQUESTED_X_MM = np.array([25, 35, 45, 55, 65, 75, 85, 95], dtype=float)

CONFIG = {
    'central_exclusion_y_min_mm': 0.65,
    'central_exclusion_y_max_mm': 1.35,
    'central_prior_min_mm': 0.65,
    'central_prior_max_mm': 1.35,
    'threshold_mad_multiplier': 2.5,
    'min_spread_um': 0.75,
    'min_baseline_samples': 30,
    'smooth_window_samples': 9,
    'min_run_samples_for_smoothing': 15,
    'min_component_samples': 6,
    'min_component_width_mm': 0.035,
    'max_component_width_mm': 1.10,
    'score_height_weight': 2.0,
    'score_width_weight': 1.0,
    'score_center_weight': 1.5,
    'score_continuity_weight': 0.6,
    'preferred_width_mm': 0.32,
    'width_scale_mm': 0.30,
    'continuity_scale_mm': 0.18,
    'min_selected_score': 1.0,
    'ambiguity_ratio': 0.85,
    'edge_search_margin_mm': 0.08,
    'edge_min_strength_um_per_mm': 50.0,
    'overview_stride_cols': 10,
    'overview_plot_stride_cols': 10,
}

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = PROJECT_DIR / 'processed_data' / 'run_outputs' / f'03_heightmap_target_extraction_exploration_{RUN_TAG}'
DIAG_DIR = OUT_DIR / 'diagnostics'
OVERVIEW_DIR = OUT_DIR / 'overviews'
TABLE_DIR = OUT_DIR / 'tables'
for d in [DIAG_DIR, OVERVIEW_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('OUT_DIR =', OUT_DIR)
print('TRACK_IDS =', TRACK_IDS)
print('REQUESTED_X_MM =', REQUESTED_X_MM.tolist())
display(pd.Series(CONFIG, name='value').to_frame())

## Helper functions

The exploratory code below intentionally does not rewrite the organizer loader. It only implements local analysis helpers for finite-run handling, substrate-focused detrending, adaptive candidate segmentation, component scoring, and gradient refinement.

In [ ]:
def robust_mad(values, scale=True):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return np.nan
    med = np.nanmedian(v)
    mad = np.nanmedian(np.abs(v - med))
    return float(1.4826 * mad if scale else mad)

def finite_runs(mask):
    idx = np.flatnonzero(np.asarray(mask, dtype=bool))
    if idx.size == 0:
        return []
    breaks = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[breaks + 1]]
    stops = np.r_[idx[breaks] + 1, idx[-1] + 1]
    return [(int(a), int(b)) for a, b in zip(starts, stops)]

def smooth_within_finite_runs(z, valid, window, min_run):
    z = np.asarray(z, dtype=float)
    out = np.full_like(z, np.nan, dtype=float)
    w = int(window)
    if w < 3 or w % 2 == 0:
        w = max(3, w + 1)
    kernel = np.ones(w, dtype=float) / w
    for a, b in finite_runs(valid):
        run = z[a:b]
        if (b - a) >= min_run:
            pad = w // 2
            padded = np.pad(run, pad_width=pad, mode='edge')
            out[a:b] = np.convolve(padded, kernel, mode='valid')
        else:
            out[a:b] = run
    return out

def gradients_within_finite_runs(y, z_smooth, valid, min_run):
    grad = np.full_like(z_smooth, np.nan, dtype=float)
    for a, b in finite_runs(valid & np.isfinite(z_smooth)):
        if (b - a) >= min_run:
            grad[a:b] = np.gradient(z_smooth[a:b], y[a:b])
    return grad

def fit_substrate_focused_plane(Z_mm, x_mm, y_mm, config, stride_x=40, stride_y=2):
    ylo = config['central_exclusion_y_min_mm']
    yhi = config['central_exclusion_y_max_mm']
    Zs = Z_mm[::stride_y, ::stride_x]
    xs = x_mm[::stride_x]
    ys = y_mm[::stride_y]
    Xs, Ys = np.meshgrid(xs, ys)
    z = Zs.ravel()
    A = np.c_[Xs.ravel(), Ys.ravel(), np.ones(Xs.size)]
    substrate = (Ys.ravel() < ylo) | (Ys.ravel() > yhi)
    valid = np.isfinite(z) & substrate
    n_initial = int(valid.sum())
    if n_initial < 100:
        return Z_mm.copy(), None, {'n_initial': n_initial, 'n_final': 0, 'excluded_y_band': [ylo, yhi]}
    keep = valid.copy()
    coef = None
    for _ in range(4):
        coef, *_ = np.linalg.lstsq(A[keep], z[keep], rcond=None)
        resid = z - A @ coef
        rv = resid[keep]
        med = np.nanmedian(rv)
        spread = max(robust_mad(rv), 1e-9)
        keep_new = valid & (np.abs(resid - med) <= 3.0 * spread)
        if keep_new.sum() < 100:
            break
        keep = keep_new
    plane = coef[0] * x_mm[None, :] + coef[1] * y_mm[:, None] + coef[2]
    return Z_mm - plane, coef, {'n_initial': n_initial, 'n_final': int(keep.sum()), 'excluded_y_band': [ylo, yhi]}

def substrate_residual_stats(Zd_um, y, config):
    sub = (y < config['central_exclusion_y_min_mm']) | (y > config['central_exclusion_y_max_mm'])
    vals = Zd_um[sub, :].ravel()
    vals = vals[np.isfinite(vals)]
    return {'substrate_residual_median_um': float(np.nanmedian(vals)), 'substrate_residual_mad_um': robust_mad(vals), 'n_substrate_residual_samples': int(vals.size)}

def baseline_threshold(y, z_um, valid, config):
    sub = valid & ((y < config['central_exclusion_y_min_mm']) | (y > config['central_exclusion_y_max_mm']))
    source = 'substrate_outside_exclusion'
    if int(sub.sum()) < config['min_baseline_samples']:
        sub = valid.copy()
        source = 'all_finite_fallback'
    vals = z_um[sub]
    if vals.size == 0:
        return np.nan, np.nan, np.nan, source
    base = float(np.nanmedian(vals))
    spread = max(float(robust_mad(vals)), config['min_spread_um'])
    thr = base + config['threshold_mad_multiplier'] * spread
    return base, spread, float(thr), source

def candidate_components(y, z_um, valid, threshold, baseline, config):
    cand = valid & np.isfinite(z_um) & (z_um >= threshold)
    comps = []
    for a, b in finite_runs(cand):
        n = b - a
        ymin = float(y[a])
        ymax = float(y[b - 1])
        width = float(ymax - ymin)
        if n < config['min_component_samples'] or width < config['min_component_width_mm'] or width > config['max_component_width_mm']:
            continue
        vals = z_um[a:b]
        comps.append({
            'a': a, 'b': b, 'n': int(n), 'y_min': ymin, 'y_max': ymax, 'width_mm': width,
            'centroid_mm': float(np.nanmean(y[a:b])),
            'median_above_baseline_um': float(np.nanmedian(vals - baseline)),
            'peak_above_baseline_um': float(np.nanmax(vals - baseline)),
        })
    return cand, comps

def score_components(comps, prev_center, config):
    prior_center = 0.5 * (config['central_prior_min_mm'] + config['central_prior_max_mm'])
    prior_half = 0.5 * (config['central_prior_max_mm'] - config['central_prior_min_mm'])
    scored = []
    for c in comps:
        center_norm = min(abs(c['centroid_mm'] - prior_center) / max(prior_half, 1e-9), 2.0)
        center_score = max(0.0, 1.0 - 0.5 * center_norm)
        height_score = math.tanh(max(c['median_above_baseline_um'], 0.0) / 12.0)
        width_score = math.exp(-((c['width_mm'] - config['preferred_width_mm']) / config['width_scale_mm']) ** 2)
        if prev_center is None or not np.isfinite(prev_center):
            continuity_score = 0.5
        else:
            continuity_score = math.exp(-abs(c['centroid_mm'] - prev_center) / config['continuity_scale_mm'])
        score = (config['score_height_weight'] * height_score +
                 config['score_width_weight'] * width_score +
                 config['score_center_weight'] * center_score +
                 config['score_continuity_weight'] * continuity_score)
        cc = dict(c)
        cc.update({'score': float(score), 'center_score': center_score, 'height_score': height_score, 'width_score': width_score, 'continuity_score': continuity_score})
        scored.append(cc)
    scored.sort(key=lambda d: d['score'], reverse=True)
    if not scored:
        return None, np.nan, scored, 'no_valid_component'
    best = scored[0]
    second = scored[1]['score'] if len(scored) > 1 else np.nan
    ambiguity = float(second / best['score']) if np.isfinite(second) and best['score'] > 0 else 0.0
    if best['score'] < config['min_selected_score']:
        return best, ambiguity, scored, 'low_score'
    if np.isfinite(ambiguity) and ambiguity >= config['ambiguity_ratio']:
        return best, ambiguity, scored, 'ambiguous'
    return best, ambiguity, scored, 'selected'

def refine_edges(y, z_um, valid, comp, config):
    z_s = smooth_within_finite_runs(z_um, valid, config['smooth_window_samples'], config['min_run_samples_for_smoothing'])
    grad = gradients_within_finite_runs(y, z_s, valid, config['min_run_samples_for_smoothing'])
    margin = config['edge_search_margin_mm']
    out = {'refined_left_mm': np.nan, 'refined_right_mm': np.nan, 'left_edge_strength': np.nan, 'right_edge_strength': np.nan, 'grad': grad, 'z_smooth': z_s}
    if comp is None:
        return out
    left_zone = (y >= comp['y_min'] - margin) & (y <= comp['y_min'] + margin) & np.isfinite(grad)
    right_zone = (y >= comp['y_max'] - margin) & (y <= comp['y_max'] + margin) & np.isfinite(grad)
    if left_zone.any():
        idx = np.flatnonzero(left_zone)
        j = idx[int(np.nanargmax(grad[idx]))]
        strength = float(grad[j])
        if strength >= config['edge_min_strength_um_per_mm']:
            out['refined_left_mm'] = float(y[j])
        out['left_edge_strength'] = strength
    if right_zone.any():
        idx = np.flatnonzero(right_zone)
        j = idx[int(np.nanargmin(grad[idx]))]
        strength = float(abs(grad[j]))
        if strength >= config['edge_min_strength_um_per_mm']:
            out['refined_right_mm'] = float(y[j])
        out['right_edge_strength'] = strength
    return out

def analyze_cross_section(track_id, x_req, x_sel, y, z_um, method, config, prev_center=None):
    valid = np.isfinite(z_um)
    finite_fraction = float(valid.mean())
    baseline, spread, threshold, baseline_source = baseline_threshold(y, z_um, valid, config)
    cand_mask, comps = candidate_components(y, z_um, valid, threshold, baseline, config) if np.isfinite(threshold) else (np.zeros_like(valid), [])
    best, ambiguity, scored, status = score_components(comps, prev_center, config)
    allow_boundaries = status == 'selected'
    edge = refine_edges(y, z_um, valid, best if allow_boundaries else None, config)
    row = {
        'track_id': int(track_id), 'requested_x_mm': float(x_req), 'selected_x_mm': float(x_sel), 'finite_fraction': finite_fraction,
        'detrend_method': method, 'baseline_um': baseline, 'robust_spread_um': spread, 'threshold_um': threshold,
        'baseline_source': baseline_source, 'fraction_finite_above_threshold': float(cand_mask.sum() / max(valid.sum(), 1)),
        'n_candidate_components': int(len(scored)), 'selected_component_score': np.nan if best is None else best['score'],
        'ambiguity_score': ambiguity, 'segmentation_left_mm': np.nan, 'segmentation_right_mm': np.nan, 'segmentation_width_mm': np.nan,
        'refined_left_mm': np.nan, 'refined_right_mm': np.nan, 'refined_width_mm': np.nan,
        'left_edge_strength': edge['left_edge_strength'], 'right_edge_strength': edge['right_edge_strength'], 'extraction_status': status,
    }
    if allow_boundaries and best is not None:
        row['segmentation_left_mm'] = best['y_min']
        row['segmentation_right_mm'] = best['y_max']
        row['segmentation_width_mm'] = best['width_mm']
        row['refined_left_mm'] = edge['refined_left_mm']
        row['refined_right_mm'] = edge['refined_right_mm']
        if np.isfinite(edge['refined_left_mm']) and np.isfinite(edge['refined_right_mm']) and edge['refined_right_mm'] > edge['refined_left_mm']:
            row['refined_width_mm'] = edge['refined_right_mm'] - edge['refined_left_mm']
    return row, {'valid': valid, 'candidate_mask': cand_mask, 'components': scored, 'selected': best if allow_boundaries else None, 'edge': edge}

def plot_diagnostic(track_id, x_req, x_sel, y, z_by_method, rows_by_method, detail_by_method, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
    for ax, method in zip(axes, ['organizer_global', 'substrate_focused']):
        z = z_by_method[method]
        row = rows_by_method[method]
        det = detail_by_method[method]
        valid = det['valid']
        ax.plot(y[valid], z[valid], '.', ms=3, color='0.25', label='finite z')
        for a, b in finite_runs(~valid):
            ax.axvspan(y[a], y[b-1], color='lightgray', alpha=0.55, lw=0)
        ax.axhline(row['baseline_um'], color='tab:blue', lw=1.2, label='baseline')
        ax.axhline(row['threshold_um'], color='tab:orange', lw=1.2, ls='--', label='threshold')
        for comp in det['components']:
            ax.axvspan(comp['y_min'], comp['y_max'], color='gold', alpha=0.18, lw=0)
        sel = det['selected']
        if sel is not None:
            ax.axvspan(sel['y_min'], sel['y_max'], color='tab:green', alpha=0.24, lw=0, label='selected component')
            ax.axvline(sel['y_min'], color='tab:green', lw=1.5, label='seg left/right')
            ax.axvline(sel['y_max'], color='tab:green', lw=1.5)
        if np.isfinite(row['refined_left_mm']):
            ax.axvline(row['refined_left_mm'], color='tab:red', lw=1.7, ls='--', label='refined left/right')
        if np.isfinite(row['refined_right_mm']):
            ax.axvline(row['refined_right_mm'], color='tab:red', lw=1.7, ls='--')
        ax.set_title(f"{method}\nTrack {track_id} requested {x_req:.0f} mm, selected {x_sel:.6f} mm\nfinite={row['finite_fraction']:.2f}, status={row['extraction_status']}", fontsize=9)
        ax.set_xlabel('physical y (mm)')
        ax.grid(alpha=0.2)
    axes[0].set_ylabel('detrended z (µm)')
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=8)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    fig.savefig(out_path, dpi=170, bbox_inches='tight')
    plt.close(fig)

def process_overview(track_id, x, y, Zd_um, method, config):
    stride = config['overview_stride_cols']
    rows = []
    selected_mask = np.zeros_like(Zd_um, dtype=bool)
    candidate_mask_plot = np.zeros_like(Zd_um[:, ::config['overview_plot_stride_cols']], dtype=bool)
    x_plot = x[::config['overview_plot_stride_cols']]
    prev_center = None
    plot_col_map = {int(j): k for k, j in enumerate(range(0, Zd_um.shape[1], config['overview_plot_stride_cols']))}
    for j in range(0, Zd_um.shape[1], stride):
        row, det = analyze_cross_section(track_id, float(x[j]), float(x[j]), y, Zd_um[:, j], method, config, prev_center)
        row['native_col'] = int(j)
        rows.append(row)
        if det['selected'] is not None:
            selected_mask[det['selected']['a']:det['selected']['b'], j] = True
            prev_center = det['selected']['centroid_mm']
        if j in plot_col_map:
            candidate_mask_plot[:, plot_col_map[j]] = det['candidate_mask']
    df = pd.DataFrame(rows)
    return df, selected_mask, candidate_mask_plot, x_plot

def plot_overview(track_id, x, y, finite_mask, candidate_mask_plot, selected_mask, x_plot, overview_df, out_path):
    fig, axes = plt.subplots(5, 1, figsize=(13, 11), sharex=False, constrained_layout=True)
    extent_full = [float(x[0]), float(x[-1]), float(y[-1]), float(y[0])]
    extent_plot = [float(x_plot[0]), float(x_plot[-1]), float(y[-1]), float(y[0])]
    axes[0].imshow(finite_mask, aspect='auto', interpolation='nearest', extent=extent_full, cmap='gray_r')
    axes[0].set_title(f'Track {track_id}: finite-value mask')
    axes[1].imshow(candidate_mask_plot, aspect='auto', interpolation='nearest', extent=extent_plot, cmap='magma')
    axes[1].set_title('Adaptive candidate elevated mask (plot stride)')
    axes[2].imshow(selected_mask, aspect='auto', interpolation='nearest', extent=extent_full, cmap='Greens')
    axes[2].set_title('Selected component/band at overview stride')
    ok = overview_df['extraction_status'].eq('selected')
    axes[2].plot(overview_df.loc[ok, 'selected_x_mm'], overview_df.loc[ok, 'segmentation_left_mm'], 'r.', ms=2, label='left')
    axes[2].plot(overview_df.loc[ok, 'selected_x_mm'], overview_df.loc[ok, 'segmentation_right_mm'], 'b.', ms=2, label='right')
    axes[2].legend(loc='upper right')
    axes[3].plot(overview_df['selected_x_mm'], overview_df['segmentation_width_mm'], '.', ms=2, label='seg width')
    axes[3].plot(overview_df['selected_x_mm'], overview_df['refined_width_mm'], '.', ms=2, label='refined width')
    axes[3].set_ylabel('width (mm)')
    axes[3].set_title('Width curve')
    axes[3].legend()
    conf = (1 - overview_df['ambiguity_score'].clip(0, 1)).fillna(0) * overview_df['finite_fraction']
    axes[4].plot(overview_df['selected_x_mm'], conf, '.', ms=2, label='confidence proxy')
    axes[4].plot(overview_df['selected_x_mm'], overview_df['ambiguity_score'], '.', ms=2, label='ambiguity')
    axes[4].set_ylim(-0.05, 1.05)
    axes[4].set_xlabel('actual x (mm)')
    axes[4].set_title('Confidence/ambiguity diagnostics')
    axes[4].legend()
    for ax in axes[:3]:
        ax.set_ylabel('physical y (mm)')
    fig.savefig(out_path, dpi=170, bbox_inches='tight')
    plt.close(fig)

print('Helper functions loaded.')

## Load height maps and compare detrending

The substrate-focused detrend is exploratory and local to this notebook. It excludes a broad fixed central y band, shared for all Tracks 8, 10, and 14, before fitting the plane. This is a conservative prior based on the previous autopsy: the visible processed band usually lies near y≈0.8–1.3 mm, while the exclusion band is widened to 0.65–1.35 mm to avoid assuming exact boundaries.

In [ ]:
height_data = {}
detrend_rows = []
for track_id in TRACK_IDS:
    print(f'Loading Track {track_id} with organizer loader...')
    h = load_wyko_asc(HEIGHT_DIR, track_id, crop_to_common=True)
    x = np.asarray(h['x_actual_mm'], dtype=float)
    y = np.asarray(h['y_mm'], dtype=float)
    Z = np.asarray(h['Z_mm'], dtype=float)

    Z_org, coef_org = robust_plane_detrend(Z, x, y)
    Z_sub, coef_sub, sub_meta = fit_substrate_focused_plane(Z, x, y, CONFIG)

    org_stats = substrate_residual_stats(Z_org * 1000.0, y, CONFIG)
    sub_stats = substrate_residual_stats(Z_sub * 1000.0, y, CONFIG)

    height_data[track_id] = {'raw': h, 'x': x, 'y': y, 'Z_mm': Z, 'organizer_global': Z_org, 'substrate_focused': Z_sub, 'coef_org': coef_org, 'coef_sub': coef_sub, 'sub_meta': sub_meta}
    detrend_rows.append({'track_id': track_id, 'detrend_method': 'organizer_global', 'coef_x': coef_org[0], 'coef_y': coef_org[1], 'coef_intercept': coef_org[2], 'finite_fit_samples_used': np.nan, 'excluded_y_band': None, **org_stats})
    detrend_rows.append({'track_id': track_id, 'detrend_method': 'substrate_focused', 'coef_x': coef_sub[0], 'coef_y': coef_sub[1], 'coef_intercept': coef_sub[2], 'finite_fit_samples_used': sub_meta['n_final'], 'excluded_y_band': sub_meta['excluded_y_band'], **sub_stats})

detrend_df = pd.DataFrame(detrend_rows)
display(detrend_df)
detrend_df.to_csv(TABLE_DIR / 'detrending_comparison.csv', index=False)
print('Saved detrending comparison:', TABLE_DIR / 'detrending_comparison.csv')

## Cross-section extraction diagnostics and overview plots

For each requested x, the notebook reports the exact nearest available height-map x column. No global NaN filling is performed. Smoothing and gradients are computed only within contiguous finite y-runs.

In [ ]:
all_rows = []
diagnostic_paths = []
overview_paths = []
overview_summaries = []

for track_id in TRACK_IDS:
    bundle = height_data[track_id]
    x, y = bundle['x'], bundle['y']
    nearest_cols = [int(np.argmin(np.abs(x - xr))) for xr in REQUESTED_X_MM]
    prev_center_by_method = {'organizer_global': None, 'substrate_focused': None}

    for x_req, j in zip(REQUESTED_X_MM, nearest_cols):
        x_sel = float(x[j])
        z_by_method = {}
        rows_by_method = {}
        detail_by_method = {}
        for method in ['organizer_global', 'substrate_focused']:
            z_um = bundle[method][:, j] * 1000.0
            row, detail = analyze_cross_section(track_id, float(x_req), x_sel, y, z_um, method, CONFIG, prev_center_by_method[method])
            all_rows.append(row)
            rows_by_method[method] = row
            detail_by_method[method] = detail
            z_by_method[method] = z_um
            if detail['selected'] is not None:
                prev_center_by_method[method] = detail['selected']['centroid_mm']
        out_path = DIAG_DIR / f'track_{track_id}_xreq_{int(x_req):03d}_diagnostic.png'
        plot_diagnostic(track_id, float(x_req), x_sel, y, z_by_method, rows_by_method, detail_by_method, out_path)
        diagnostic_paths.append(out_path)

    Zd_um = bundle['substrate_focused'] * 1000.0
    overview_df, selected_mask, candidate_mask_plot, x_plot = process_overview(track_id, x, y, Zd_um, 'substrate_focused', CONFIG)
    overview_df.to_csv(TABLE_DIR / f'track_{track_id}_overview_substrate_focused.csv', index=False)
    finite_mask = np.isfinite(bundle['Z_mm'])
    overview_path = OVERVIEW_DIR / f'track_{track_id}_overview_substrate_focused.png'
    plot_overview(track_id, x, y, finite_mask, candidate_mask_plot, selected_mask, x_plot, overview_df, overview_path)
    overview_paths.append(overview_path)
    overview_summaries.append({'track_id': track_id, 'overview_rows': len(overview_df), **overview_df['extraction_status'].value_counts().to_dict()})

method_df = pd.DataFrame(all_rows)
method_df.to_csv(TABLE_DIR / 'representative_x_method_comparison.csv', index=False)
display(method_df)
print(f'Saved {len(diagnostic_paths)} diagnostic figures to {DIAG_DIR}')
print(f'Saved {len(overview_paths)} overview figures to {OVERVIEW_DIR}')
print('Saved method table:', TABLE_DIR / 'representative_x_method_comparison.csv')
display(pd.DataFrame(overview_summaries).fillna(0))

## Critical self-audit

This cell is populated after the notebook is executed end-to-end.

In [ ]:
success_counts = method_df.groupby(['track_id', 'detrend_method', 'extraction_status']).size().unstack(fill_value=0)
suspicious = method_df.assign(
    missing_penalty=1-method_df['finite_fraction'],
    ambiguity_filled=method_df['ambiguity_score'].fillna(1),
    edge_fail=((method_df['refined_left_mm'].isna()) | (method_df['refined_right_mm'].isna())).astype(int),
)
suspicious['suspicion_score'] = suspicious['missing_penalty'] + suspicious['ambiguity_filled'] + 0.25*suspicious['edge_fail'] + suspicious['extraction_status'].ne('selected').astype(float)
suspicious_top10 = suspicious.sort_values('suspicion_score', ascending=False).head(10)

display(Markdown('### Extraction status counts'))
display(success_counts)
display(Markdown('### Ten most suspicious representative cases'))
display(suspicious_top10[['track_id','requested_x_mm','selected_x_mm','detrend_method','finite_fraction','n_candidate_components','ambiguity_score','extraction_status','suspicion_score']])

summary = {
    'out_dir': str(OUT_DIR),
    'diagnostic_count': len(diagnostic_paths),
    'overview_count': len(overview_paths),
    'status_counts': success_counts.reset_index().to_dict(orient='records'),
    'suspicious_top10': suspicious_top10[['track_id','requested_x_mm','selected_x_mm','detrend_method','finite_fraction','n_candidate_components','ambiguity_score','extraction_status','suspicion_score']].to_dict(orient='records'),
    'config': CONFIG,
}
(TABLE_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2))
print('Saved run summary:', TABLE_DIR / 'run_summary.json')

## Critical self-audit answers

1. **Does substrate-focused detrending visibly improve separation over organizer detrending?** Partly. It improved representative extraction counts from 5/24 selected with organizer global detrending to 10/24 selected with substrate-focused detrending. It also reduced substrate residual median and MAD for Tracks 8 and 10. Track 14 did not clearly improve in robust spread, so B is not automatically superior.

2. **At which track/x cases does adaptive thresholding fail?** It fails hard at Track 10 x≈25 mm because the selected column has no finite values. It also produces no valid component at Track 8 x≈25, 55, 95; Track 10 x≈95; and Track 14 x≈25, 45, 75, 95 under substrate-focused detrending.

3. **At which cases does component selection become ambiguous?** Under substrate-focused detrending: Track 8 x≈45; Track 10 x≈35, 45, and 85. Organizer-global detrending also shows ambiguity at Track 8 x≈45, Track 10 x≈85, and Track 14 x≈85.

4. **At which cases does gradient refinement help?** It fully refined both boundaries for substrate-focused Track 8 x≈35 and x≈75. It also provided one-sided edge evidence in several selected cases, but did not reliably produce both boundaries.

5. **At which cases does gradient refinement fail because of missing data?** Many selected cases have one or both refined boundaries missing despite a segmentation component: Track 8 x≈65 and 85; Track 10 x≈55 and 75; Track 14 x≈35, 55, 65, and 85. These failures are consistent with finite-run fragmentation, weak edge gradients, or missing edge neighborhoods.

6. **Are the same parameters plausibly usable for Tracks 8, 10, and 14?** A single parameter set produced usable selected components on all three tracks, but not uniformly. Track 10 remains the weakest case with 2 selected, 3 ambiguous, and 3 no-valid representative substrate-focused cases.

7. **Is there evidence that the current method is relying too heavily on a fixed central-y prior?** Yes. The scoring includes a central-y prior and many successful components lie near that band. This is helpful against edge artifacts but scientifically arbitrary; sensitivity analysis must test weaker priors and continuity-only alternatives.

8. **Which 5–10 diagnostic cases should a human inspect most carefully before freezing the target extractor?** Track 10 x≈25; Track 10 x≈35; Track 8 x≈25; Track 8 x≈45; Track 10 x≈45; Track 10 x≈65; Track 10 x≈75; Track 14 x≈85; Track 14 x≈25; Track 14 x≈95.

9. **What exact algorithm is recommended for production after this experiment?** Recommended production direction: substrate-focused plane detrend; per-cross-section/local-window robust substrate baseline; adaptive threshold relative to local MAD; finite-run-preserving connected components; continuity-aware component scoring; conservative status labels (`selected`, `ambiguous`, `no_valid_component`); gradient refinement only as optional boundary adjustment with independent left/right failure; export segmentation boundaries when gradient evidence is weak and record confidence/edge diagnostics.

10. **Which thresholds/weights remain scientifically arbitrary and need sensitivity analysis?** Central exclusion/prior band 0.65–1.35 mm, threshold multiplier 2.5×MAD, minimum component width 0.035 mm, maximum component width 1.10 mm, preferred width 0.32 mm, scoring weights, ambiguity ratio 0.85, edge search margin 0.08 mm, and edge strength threshold 50 µm/mm.